# 25 · Frozen evaluation cohort: CBIS-DDSM patients never used in the study (CPU, about 2–3 h, resumable)
**Runtime: CPU.** No model is loaded here. Read-only for all existing data and runs. Writes `data/processed/cbis_mass_256_jit_frozen/`
and the protocol file `results/p2_rev4/PROTOCOL_rev4_prespecified.txt` (written before anything else, with a UTC time stamp).

Cohort rule (prespecified, cell 1): all CBIS-DDSM mass ROIs with an available full-field image, of patients **not** among the 208 study
patients; patients shuffled with `random.Random(2026)` and added whole until 400 ROIs. Crops: exactly the study's randomized protocol.
Then the two no-network baselines (mean training mask, centred disc of each fold) on the frozen crops.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, time, gc, hashlib, random
import numpy as np, pandas as pd
PROJECT  = '/content/drive/MyDrive/quantum-placement'
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
FROZ_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit_frozen'
DMID_DIR = f'{PROJECT}/data/external/dmid_mass_256_jit_v2/roi'
OUT      = f'{PROJECT}/results/p2_rev4'
ROOTS    = {'train': f'{PROJECT}/results/p2_jit_runs_train', 'val': f'{PROJECT}/results/p2_jit_runs_val',
            'train4': f'{PROJECT}/results/p2_rev4_runs_train', 'val4': f'{PROJECT}/results/p2_rev4_runs_val'}
SCHED    = {'train': 'train', 'val': 'val', 'train4': 'train', 'val4': 'val'}
for d in [OUT] + list(ROOTS.values()): os.makedirs(d, exist_ok=True)
PROTO_FILE = f'{OUT}/PROTOCOL_rev4_prespecified.txt'
PROTO = """Paper 2 - revision 4 analyses, written before any of them was run (frozen cohort selected, crops made or runs trained).

A. Frozen evaluation on untouched patients. The pipeline is frozen: the 190 saved models, crop protocol, threshold 0.5 and metrics are
   unchanged. Cohort: CBIS-DDSM mass ROIs (same eligibility as the study: every mass ROI whose full-field image is available, no filtering)
   of patients NOT among the 208 study patients. Patients are shuffled once with Python random.Random(2026) and added whole, in that
   order, while the total stays at or below 400 ROIs (patients that would exceed 400 are skipped); selection stops at 400 ROIs.
   Crops: the study's randomized protocol (seed from the MD5 of the ROI identifier). Every saved model predicts every frozen crop; per
   crop and seed, Dice is averaged over the five fold models (as for DMID). Primary estimands: the nine model contrasts of Tables VI-VII
   under each schedule, crossed hierarchical bootstrap over patients and seeds (10,000 replicates, rng(0)), Holm within the nine
   contrasts of each schedule. Secondary: model levels, the two no-network baselines, empty-prediction rates, HD95.
B. Inference checks (no training). (i) Exact inference repeated three times within one session (same checkpoint, same inputs) and
   compared with the stored values, for the 60 circuit runs and their Dense(4) partners, on the CBIS-DDSM test fold and DMID.
   (ii) Initialization audit: for seeds 0-2 and fold 0, the global and patch-wise models (circuit, Dense(4), trigonometric, identity)
   are built with the training seed; the weights of the layers shared by all variants are compared, and the initial block input and
   output statistics and the gradient norms at the block are recorded on four training batches.
C. New training runs (training and evaluation exactly as in the study):
   C. Trigonometric control g_tr, p_tr: z = tanh(A cos((pi/2) x + phi)), A 4x4 Glorot uniform, phi uniform [0, 2 pi): 20 parameters,
       same slot as the circuit; seeds 0-2, folds 0-4, both schedules (60 runs).
   Primary estimands (CBIS-DDSM test folds): circuit - trigonometric at each placement under each schedule (4 contrasts, Holm over
   the four). Secondary: trigonometric - Dense(4) and trigonometric - identity at each placement; the same contrasts on DMID
   (exploratory) and on the frozen cohort; code-replacement drops of the trigonometric models.
   No further seeds and no further learning-rate schedule are trained.
All runs and all results are reported; none is discarded."""
if not os.path.exists(PROTO_FILE):
    open(PROTO_FILE, 'w').write(PROTO + '\nwritten (UTC): ' + time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()) + '\n')
print('protocol:', open(PROTO_FILE).read().splitlines()[-1])
CODE_DIR = f'{PROJECT}/code'
if not os.path.exists(f'{CODE_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    CODE_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p in (CODE_DIR, f'{PROJECT}/code'):
    if p not in sys.path: sys.path.insert(0, p)


In [ ]:
# 1 · Helpers: identical to notebook 11 (TCIA access, case table, windowing, mask selection, randomized crop)
!pip -q install pydicom
import pydicom, re, io, shutil, zipfile, requests, cv2
TCIA_API = "https://services.cancerimagingarchive.net/nbia-api/services/v1"
CROP_SIZE, SCALE_LO, SCALE_HI, SHIFT = 256, 1.5, 3.0, 0.8
RAW_TMP = '/content/raw_tmp'; os.makedirs(RAW_TMP, exist_ok=True)
for d in (f'{FROZ_DIR}/images', f'{FROZ_DIR}/masks'): os.makedirs(d, exist_ok=True)
def get_series_table(collection="CBIS-DDSM", retries=3):
    for i in range(retries):
        try:
            r = requests.get(f"{TCIA_API}/getSeries", params={"Collection": collection}, timeout=120); r.raise_for_status()
            return pd.DataFrame(r.json())
        except Exception as e:
            print(f"  getSeries failed ({e}); retry {i + 1}/{retries}"); time.sleep(5 * (i + 1))
    raise RuntimeError("Could not reach TCIA")
def download_series(uid, out_dir, retries=3):
    dest = os.path.join(out_dir, uid)
    if os.path.isdir(dest) and any(f.endswith(".dcm") for f in os.listdir(dest)):
        return sorted(os.path.join(dest, f) for f in os.listdir(dest) if f.endswith(".dcm"))
    os.makedirs(dest, exist_ok=True)
    for i in range(retries):
        try:
            r = requests.get(f"{TCIA_API}/getImage", params={"SeriesInstanceUID": uid}, timeout=600); r.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(r.content)) as z: z.extractall(dest)
            break
        except Exception as e:
            print(f"  download {uid[-12:]} failed ({e}); retry {i + 1}/{retries}"); time.sleep(5 * (i + 1))
    else:
        shutil.rmtree(dest, ignore_errors=True); return []
    files = []
    for root, _, fs in os.walk(dest): files += [os.path.join(root, f) for f in fs if f.lower().endswith(".dcm")]
    return sorted(files)
ROI_PID = re.compile(r"^(Mass-(Training|Test)_P_\d+_(LEFT|RIGHT)_(CC|MLO))_(\d+)$")
FULL_PID = re.compile(r"^Mass-(Training|Test)_P_\d+_(LEFT|RIGHT)_(CC|MLO)$")
def build_case_table(series_df):
    df = series_df[series_df["PatientID"].str.startswith("Mass-")].copy()
    full = {r["PatientID"]: r["SeriesInstanceUID"] for _, r in df.iterrows() if FULL_PID.match(r["PatientID"])}
    rows = []
    for _, r in df.iterrows():
        m = ROI_PID.match(r["PatientID"])
        if not m or m.group(1) not in full: continue
        parts = m.group(1).split("_")
        rows.append(dict(roi_id=r["PatientID"], full_id=m.group(1), patient=f"P_{parts[2]}", split=m.group(2), side=m.group(3),
                         view=m.group(4), full_uid=full[m.group(1)], roi_uid=r["SeriesInstanceUID"]))
    t = pd.DataFrame(rows)
    t = t.groupby(["roi_id", "full_id", "patient", "split", "side", "view", "full_uid"], as_index=False)["roi_uid"].agg(list)
    return t.sort_values("roi_id").reset_index(drop=True)
def to_uint8(img):
    img = img.astype(np.float32); nz = img[img > 0]
    lo, hi = (np.percentile(nz, 1), np.percentile(nz, 99.5)) if nz.size else (img.min(), img.max())
    return (np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)
def is_mask(arr):
    return np.unique(arr[:: max(1, arr.shape[0] // 256), :: max(1, arr.shape[1] // 256)]).size <= 2
def pick_mask(arrays, full_shape):
    cands = [a for a in arrays if is_mask(a)]
    if not cands: return None
    m = (max(cands, key=lambda a: a.size) > 0).astype(np.uint8)
    if m.shape != tuple(full_shape): m = cv2.resize(m, (full_shape[1], full_shape[0]), interpolation=cv2.INTER_NEAREST)
    return m
def jit_crop(img, mask, rid, size=CROP_SIZE):
    ys, xs = np.where(mask > 0)
    if ys.size == 0: return None, None, None
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    L = max(y1 - y0, x1 - x0)
    rng = np.random.default_rng(int(hashlib.md5(rid.encode()).hexdigest()[:8], 16))
    scale = rng.uniform(SCALE_LO, SCALE_HI); side = max(int(L * scale), 64)
    slack_y, slack_x = (side - (y1 - y0)) / 2, (side - (x1 - x0)) / 2
    dy, dx = rng.uniform(-SHIFT, SHIFT, 2)
    cy = int(round((y0 + y1) / 2 + dy * slack_y)); cx = int(round((x0 + x1) / 2 + dx * slack_x))
    H, W = mask.shape; top, left = cy - side // 2, cx - side // 2
    pad = max(0, -top, -left, top + side - H, left + side - W)
    if pad:
        img = np.pad(img, pad, mode='constant'); mask = np.pad(mask, pad, mode='constant'); top += pad; left += pad
    ic = cv2.resize(img[top:top + side, left:left + side], (size, size), interpolation=cv2.INTER_AREA)
    mc = cv2.resize(mask[top:top + side, left:left + side].astype(np.uint8), (size, size), interpolation=cv2.INTER_NEAREST)
    mc = (mc > 0).astype(np.uint8) * 255
    return ic, mc, dict(bbox_h=int(y1 - y0), bbox_w=int(x1 - x0), crop_side=side, scale=round(float(scale), 4),
                        shift_y=round(float(dy), 4), shift_x=round(float(dx), 4), padded=bool(pad), mask_frac=round(float((mc > 0).mean()), 4))
def read_dicoms(files):
    out = []
    for f in files:
        try: out.append(pydicom.dcmread(f).pixel_array)
        except Exception as e: print('   unreadable', os.path.basename(f), e)
    return out
print('helpers ready')

In [ ]:
# 2 · Frozen cohort selection (prespecified rule; never looks at images, outcomes or models)
COH = f'{FROZ_DIR}/cohort.csv'
if not os.path.exists(COH):
    cases = build_case_table(get_series_table())
    study = pd.read_csv(f'{CBIS_DIR}/folds.csv'); study_pat = set(study.patient)
    print('mass ROIs with full image:', len(cases), '| patients:', cases.patient.nunique(), '| study ROIs found:', cases.roi_id.isin(study.roi_id).sum())
    assert cases.roi_id.isin(study.roi_id).sum() == 400 and len(study_pat) == 208
    rest = cases[~cases.patient.isin(study_pat)].reset_index(drop=True)
    pats = sorted(rest.patient.unique()); random.Random(2026).shuffle(pats)
    keep, n = [], 0
    for p in pats:
        k = int((rest.patient == p).sum())
        if n + k > 400: continue
        keep.append(p); n += k
        if n == 400: break
    coh = rest[rest.patient.isin(keep)].copy(); coh['roi_uid'] = coh.roi_uid.apply(json.dumps)
    coh.to_csv(COH, index=False)
coh = pd.read_csv(COH); coh['roi_uid'] = coh.roi_uid.apply(json.loads)
assert not set(coh.patient) & set(pd.read_csv(f'{CBIS_DIR}/folds.csv').patient), 'overlap with study patients'
print('frozen cohort:', len(coh), 'ROIs |', coh.patient.nunique(), 'patients |', coh.split.value_counts().to_dict())

In [ ]:
# 3 · Download + randomized crops (resumable)
from tqdm.auto import tqdm
meta_csv = f'{FROZ_DIR}/metadata.csv'
meta = pd.read_csv(meta_csv).to_dict('records') if os.path.exists(meta_csv) else []
done = {m['roi_id'] for m in meta}; failed = []
for full_id, grp in tqdm(coh.groupby('full_id'), desc='images'):
    todo = grp[~grp.roi_id.isin(done)]
    if todo.empty: continue
    arrs = read_dicoms(download_series(grp.full_uid.iloc[0], RAW_TMP))
    if not arrs:
        failed += list(todo.roi_id); shutil.rmtree(RAW_TMP, ignore_errors=True); os.makedirs(RAW_TMP); continue
    full = max(arrs, key=lambda a: a.size); full8 = to_uint8(full)
    for _, r in todo.iterrows():
        a = []
        for uid in r.roi_uid: a += read_dicoms(download_series(uid, RAW_TMP))
        mask = pick_mask(a, full.shape)
        ic, mc, info = jit_crop(full8, mask, r.roi_id) if mask is not None else (None, None, None)
        if ic is None: failed.append(r.roi_id); continue
        cv2.imwrite(f'{FROZ_DIR}/images/{r.roi_id}.png', ic); cv2.imwrite(f'{FROZ_DIR}/masks/{r.roi_id}.png', mc)
        meta.append(dict(roi_id=r.roi_id, patient=r.patient, split=r.split, side=r.side, view=r.view, **info)); done.add(r.roi_id)
    pd.DataFrame(meta).to_csv(meta_csv, index=False)
    shutil.rmtree(RAW_TMP, ignore_errors=True); os.makedirs(RAW_TMP)
m = pd.DataFrame(meta).sort_values('roi_id').reset_index(drop=True); m.to_csv(meta_csv, index=False)
print(f'frozen crops: {len(m)} / {len(coh)} | failed (no readable, non-empty mask): {failed}')
print(m[['scale', 'mask_frac']].describe().round(3).to_string())

In [ ]:
# 4 · No-network baselines of every fold (training crops of the study) on the frozen crops
def load_png(d, ids, mask):
    return np.stack([(cv2.imread(f'{d}/masks/{r}.png', 0) > 127) if mask else cv2.imread(f'{d}/images/{r}.png', 0) for r in ids])
F = pd.read_csv(f'{CBIS_DIR}/folds.csv'); YC = load_png(CBIS_DIR, F.roi_id, True)
m = pd.read_csv(meta_csv); YF = load_png(FROZ_DIR, m.roi_id, True)
def dice(Y, P):
    tp = (Y & P).sum((1, 2)); s = Y.sum((1, 2)) + P.sum((1, 2))
    return np.where(s == 0, 1.0, 2 * tp / np.maximum(s, 1))
def disc(area, n=256):
    r = np.sqrt(area / np.pi); yy, xx = np.mgrid[:n, :n]
    return (yy - (n - 1) / 2) ** 2 + (xx - (n - 1) / 2) ** 2 <= r ** 2
rows = []
for k in range(5):
    f = F.fold.values; tr = (f != k) & (f != (k + 1) % 5)
    for nm, sh in (('mean_mask', YC[tr].mean(0) > 0.5), ('centred_disc', disc(YC[tr].sum((1, 2)).mean()))):
        rows.append(pd.DataFrame(dict(baseline=nm, fold=k, roi_id=m.roi_id, patient=m.patient, dice=dice(YF, sh[None]))))
bl = pd.concat(rows, ignore_index=True); bl.to_csv(f'{OUT}/frozen_baselines_per_image.csv', index=False)
print(bl.groupby('baseline').dice.mean().round(4).to_string())
print('Done. Next: notebook 28 (evaluation of the saved models on these crops).')